# Homografia boiska do widoku bird view z `ClassicalHomographyV2`

Ten notebook prowadzi przez klasyczny pipeline estymacji homografii dla obrazu z transmisji piłkarskiej. Celem jest wyznaczenie macierzy `H`, która mapuje punkt z obrazu w pikselach do układu boiska w metrach, a potem wykonanie przejścia do widoku z góry.

Zakładany układ boiska:

* środek boiska: `(0, 0)`
* oś `x`: długość boiska, od `-52.5` do `52.5` m
* oś `y`: szerokość boiska, od `-34` do `34` m
* `H`: piksel obrazu -> metry na boisku

Pipeline w tym notebooku:

1. Wczytanie klatki
2. Segmentacja murawy
3. Ekstrakcja maski białych linii
4. Detekcja odcinków HoughLinesP
5. Scalanie kolinearnych odcinków
6. Grupowanie linii w dwa kierunki i punkty zbiegu
7. Próba automatycznej estymacji `H`
8. Fallback korespondencyjny, gdy automat ma za mało stabilnych cech
9. Nakładka modelu boiska na obraz
10. Warp obrazu do mapy bird view

## Krok 0: importy i funkcje pomocnicze

Notebook próbuje importować `classical_homography_v2.py` z katalogu notebooka, katalogu nadrzędnego oraz z `/mnt/data`. Dzięki temu powinien działać zarówno tutaj, jak i po przeniesieniu pliku `.ipynb` razem z klasą do jednego folderu.

In [ ]:
import sys
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

SEARCH_DIRS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "src" / "calibration",
    Path.cwd().parent / "src" / "calibration",
    Path("/mnt/data"),
]
for d in SEARCH_DIRS:
    if d.exists():
        sys.path.insert(0, str(d))

try:
    from classical_homography_v2 import ClassicalHomographyV2, PITCH_LENGTH_M, PITCH_WIDTH_M
except ModuleNotFoundError:
    from src.calibration.classical_homography_v2 import ClassicalHomographyV2, PITCH_LENGTH_M, PITCH_WIDTH_M

%matplotlib inline
plt.rcParams["figure.figsize"] = (16, 9)
plt.rcParams["figure.dpi"] = 110

PITCH_L = PITCH_LENGTH_M / 2.0
PITCH_W = PITCH_WIDTH_M / 2.0
CENTER_CIRCLE_R = 9.15


def read_bgr(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Nie można wczytać obrazu: {path}")
    return img


ModuleNotFoundError: No module named 'classical_homography_v2'

## Krok 1: wybór obrazu

Domyślnie ustawiam `000250.jpg`, bo ma dobrze widoczną linię środkową i koło. To bardzo dobry przypadek do pokazania przejścia na bird view z korespondencji punktowych. Możesz zmienić `DEFAULT_IMAGE_NAME` na dowolny z załączonych kadrów.

In [ ]:
IMAGE_PATHS = []
for d in SEARCH_DIRS:
    if d.exists():
        IMAGE_PATHS.extend(d.glob("*.jpg"))
IMAGE_PATHS = sorted({p.resolve() for p in IMAGE_PATHS}, key=lambda p: p.name)

print("Dostępne obrazy:")
for i, p in enumerate(IMAGE_PATHS):
    print(f"[{i}] {p.name}  {p}")

DEFAULT_IMAGE_NAME = "000250.jpg"
image_path = next((p for p in IMAGE_PATHS if p.name == DEFAULT_IMAGE_NAME), IMAGE_PATHS[0])
image = read_bgr(image_path)
h, w = image.shape[:2]
print(f"\nWybrany obraz: {image_path.name}, rozmiar: {w}x{h}")
show_bgr(image, title=f"Klatka wejściowa: {image_path.name}")

## Krok 2: inicjalizacja kalibratora

W trybie debugowania zostawiam `refine_steps = 0`, żeby szybciej widzieć wynik surowej estymacji. Gdy korespondencje albo automat dają już sensowną homografię, można zwiększyć `refine_steps`, na przykład do `300` albo `500`.

In [ ]:
RANSAC_ITERS = 800
MIN_SCORE = 0.10
REFINE_STEPS = 0

cal = ClassicalHomographyV2(
    image_width=w,
    image_height=h,
    ransac_iterations=RANSAC_ITERS,
    min_score=MIN_SCORE,
    refine_steps=REFINE_STEPS,
    verbose=True,
)

## Krok 3: segmentacja murawy

`segment_field()` tworzy maskę zielonej części boiska. Dzięki temu reklamy, trybuny i większość elementów poza polem gry nie powinny przechodzić dalej do maski linii.

In [ ]:
field_mask, hull = cal.segment_field(image)

vis_hull = image.copy()
if hull is not None:
    cv2.polylines(vis_hull, [hull], True, (0, 0, 255), 3)

show_side_by_side([
    {"img": vis_hull, "title": "Obraz z otoczką pola"},
    {"img": field_mask, "title": f"Maska murawy, pokrycie: {np.mean(field_mask > 0) * 100:.1f}%", "mask": True},
])

## Krok 4: maska białych linii

`extract_line_mask()` filtruje jasne, mało nasycone piksele w obrębie murawy, a następnie odrzuca komponenty, które wyglądają bardziej jak plamy niż cienkie linie.

In [ ]:
line_mask = cal.extract_line_mask(image, field_mask)
print(f"Liczba pikseli maski linii: {int((line_mask > 0).sum())}")
show_mask(line_mask, title="Maska białych linii")

## Krok 5: odcinki z HoughLinesP

Na masce linii wykrywamy krótkie odcinki. Na tym etapie będą widoczne także fragmenty koła środkowego, linii bocznych, linii pola karnego oraz czasem zakłócenia od białych strojów.

In [ ]:
segments = cal.detect_segments(line_mask)
print(f"Liczba odcinków HoughLinesP: {len(segments)}")
show_bgr(draw_segments(image, segments), title=f"Wykryte odcinki: {len(segments)}")

## Krok 6: scalanie odcinków kolinearnych

`merge_collinear()` scala krótkie fragmenty o podobnym kierunku i małej odległości od wspólnej prostej. Wynikiem są dłuższe linie opisane przez punkty końcowe, kąt, kierunek PCA i łączną długość.

In [ ]:
merged = cal.merge_collinear(segments)
print(f"Liczba linii po scaleniu: {len(merged)}")
for i, m in enumerate(merged):
    print(
        f"[{i:02d}] angle={m['angle']:7.2f} deg, "
        f"length={m['total_length']:8.1f}, n_segments={m['n_segments']}"
    )
show_bgr(draw_merged_lines(image, merged), title=f"Scalone linie: {len(merged)}")

## Krok 7: grupowanie w dwa kierunki i przecięcia

`group_two_directions()` dzieli linie na dwie rodziny kierunkowe. Dla pełnego automatu potrzebujemy co najmniej dwóch linii w każdej grupie, bo wtedy da się policzyć punkty zbiegu dla obu osi boiska.

In [ ]:
group_a, group_b = cal.group_two_directions(merged)
intersections = cal.detect_intersections(group_a, group_b)
vp_a = cal.compute_vanishing_point(group_a)
vp_b = cal.compute_vanishing_point(group_b)

print(f"Grupa A: {len(group_a)}")
print(f"Grupa B: {len(group_b)}")
print(f"Przecięcia linii A x B: {len(intersections)}")
print(f"VP A: {None if vp_a is None else np.round(vp_a, 3)}")
print(f"VP B: {None if vp_b is None else np.round(vp_b, 3)}")

show_bgr(
    draw_groups(image, group_a, group_b, intersections),
    title="Grupy kierunkowe: czerwone i niebieskie, przecięcia: żółte",
)

## Krok 8: próba pełnego automatu

`estimate_homography()` najpierw próbuje wariantu z kołem środkowym, a potem VP RANSAC. W wielu trudnych kadrach transmisyjnych automat może zwrócić `None`, co jest lepsze niż zwrócenie losowej, błędnej homografii.

Dla kadrów z samym kołem środkowym często brakuje drugiej rodziny prostych linii. Wtedy przechodzimy do kroku korespondencyjnego.

In [ ]:
H_auto, score_auto = None, 0.0

if len(group_a) >= 2 and len(group_b) >= 2:
    H_auto, score_auto = cal.estimate_homography(line_mask, group_a, group_b)
    print(f"Wynik automatu: H={H_auto is not None}, F1={score_auto:.3f}")
else:
    print("Za mało linii w jednej z grup. Automat VP nie ma pełnych danych.")

if H_auto is not None and score_auto >= cal.min_score:
    if cal.refine_steps > 0:
        H_auto, score_auto = cal.refine_homography(H_auto, line_mask)
    show_bgr(cal.draw_overlay(image, H_auto, thickness=2), title=f"Automat, F1={score_auto:.3f}")
    print("H_auto, piksel -> metry:")
    print(np.array2string(H_auto, precision=6, suppress_small=True))
else:
    print("Brak pewnej homografii automatycznej. Przejdź do fallbacku korespondencyjnego.")

## Krok 9: fallback z korespondencji punktowych

To nadal korzysta z klasy `ClassicalHomographyV2`, konkretnie z metody `estimate_homography_from_correspondences()`. Różnica polega na tym, że podajemy znane pary:

* punkt w obrazie `(u, v)`
* odpowiadający punkt na boisku `(x, y)` w metrach

Dla `000250.jpg` dodałem gotowy preset z punktami na kole środkowym i punktem przecięcia linii środkowej z górną linią boczną. Traktuj go jako start do korekty. Po kliknięciu dokładniejszych punktów wynik będzie lepszy.

In [ ]:
# Punkty modelu dla koła środkowego i linii środkowej.
# Uwaga: wartości pikselowe są ręcznie dobranym presetem dla 000250.jpg.
# Przy innym obrazie uzupełnij własne punkty albo użyj klikacza z kolejnej komórki.

MANUAL_PRESETS = {
    "000250.jpg": [
        # label, point_in_image_px, point_on_pitch_m
        ("center_line_top_sideline", (985.0, 220.0), (0.0, -PITCH_W)),
        ("center_circle_top",        (985.0, 375.0), (0.0, -CENTER_CIRCLE_R)),
        ("center_circle_bottom",     (985.0, 627.0), (0.0,  CENTER_CIRCLE_R)),
        ("center_circle_left",       (393.0, 500.0), (-CENTER_CIRCLE_R, 0.0)),
        ("center_circle_right",      (1579.0, 500.0), ( CENTER_CIRCLE_R, 0.0)),
    ],
}

manual_points = MANUAL_PRESETS.get(image_path.name, [])

print(f"Preset dla {image_path.name}: {len(manual_points)} punktów")
for label, img_pt, model_pt in manual_points:
    print(f"{label:28s} image={img_pt}  pitch={model_pt}")

vis_pts = image.copy()
for label, (u, v), _ in manual_points:
    cv2.circle(vis_pts, (int(round(u)), int(round(v))), 8, (0, 255, 255), -1)
    cv2.putText(vis_pts, label, (int(round(u)) + 10, int(round(v)) - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)
show_bgr(vis_pts, title="Punkty korespondencyjne na obrazie")

In [ ]:
H_manual, score_manual = None, 0.0

if len(manual_points) >= 4:
    img_pts = np.array([p[1] for p in manual_points], dtype=np.float64)
    model_pts = np.array([p[2] for p in manual_points], dtype=np.float64)

    H_manual, score_manual = cal.estimate_homography_from_correspondences(
        img_pts,
        model_pts,
        line_mask=line_mask,
        ransac_thresh=3.0,
    )

    print(f"Wynik korespondencyjny: H={H_manual is not None}, F1={score_manual:.3f}")
    if H_manual is not None:
        print("H_manual, piksel -> metry:")
        print(np.array2string(H_manual, precision=6, suppress_small=True))
        show_bgr(cal.draw_overlay(image, H_manual, thickness=2), title=f"Nakładka modelu boiska, F1={score_manual:.3f}")
else:
    print("Brak minimum 4 punktów korespondencyjnych. Uzupełnij manual_points albo użyj klikacza niżej.")

## Krok 10: interaktywny klikacz punktów

Ta komórka jest opcjonalna. W klasycznym Jupyterze możesz odkomentować `%matplotlib widget`, uruchomić `make_clicker()` i klikać punkty w kolejności z listy `CLICK_LANDMARKS`.

Po kliknięciu uruchom kolejną komórkę, która zbuduje homografię z `clicked_points`.

In [ ]:
CLICK_LANDMARKS = [
    ("center_line_top_sideline", (0.0, -PITCH_W)),
    ("center_circle_top",        (0.0, -CENTER_CIRCLE_R)),
    ("center_circle_bottom",     (0.0,  CENTER_CIRCLE_R)),
    ("center_circle_left",       (-CENTER_CIRCLE_R, 0.0)),
    ("center_circle_right",      ( CENTER_CIRCLE_R, 0.0)),
]

clicked_points = []

def make_clicker():
    clicked_points.clear()
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(bgr_to_rgb(image))
    ax.axis("on")
    ax.set_title(f"Kliknij: {CLICK_LANDMARKS[0][0]}")

    state = {"i": 0}

    def onclick(event):
        if event.xdata is None or event.ydata is None:
            return
        if state["i"] >= len(CLICK_LANDMARKS):
            return
        label, model_pt = CLICK_LANDMARKS[state["i"]]
        img_pt = (float(event.xdata), float(event.ydata))
        clicked_points.append((label, img_pt, model_pt))
        ax.plot(event.xdata, event.ydata, "yo")
        ax.text(event.xdata + 8, event.ydata - 8, label, color="yellow")
        state["i"] += 1
        if state["i"] < len(CLICK_LANDMARKS):
            ax.set_title(f"Kliknij: {CLICK_LANDMARKS[state['i']][0]}")
        else:
            ax.set_title("Gotowe. Uruchom komórkę estymacji z clicked_points.")
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", onclick)
    return fig

# W razie potrzeby odkomentuj dwie linie poniżej:
# %matplotlib widget
# make_clicker()

In [ ]:
H_clicked, score_clicked = None, 0.0

if len(clicked_points) >= 4:
    img_pts = np.array([p[1] for p in clicked_points], dtype=np.float64)
    model_pts = np.array([p[2] for p in clicked_points], dtype=np.float64)

    H_clicked, score_clicked = cal.estimate_homography_from_correspondences(
        img_pts,
        model_pts,
        line_mask=line_mask,
        ransac_thresh=3.0,
    )

    print(f"Wynik z kliknięć: H={H_clicked is not None}, F1={score_clicked:.3f}")
    if H_clicked is not None:
        show_bgr(cal.draw_overlay(image, H_clicked, thickness=2), title=f"Kliknięcia, F1={score_clicked:.3f}")
else:
    print(f"Kliknięte punkty: {len(clicked_points)}. Do estymacji potrzeba minimum 4.")

## Krok 11: wybór najlepszej homografii

Kolejność priorytetu:

1. `H_clicked`, jeśli kliknięto punkty
2. `H_manual`, jeśli istnieje preset lub ręczne punkty
3. `H_auto`, jeśli automat dał sensowny wynik

W praktyce do publikowania wyników warto zapisać osobno źródło homografii, na przykład `auto`, `manual` albo `clicked`.

In [ ]:
H_final = None
H_source = None
H_score = None

for name, H_candidate, score_candidate in [
    ("clicked", H_clicked, score_clicked),
    ("manual", H_manual, score_manual),
    ("auto", H_auto, score_auto),
]:
    if H_candidate is not None:
        H_final = H_candidate
        H_source = name
        H_score = score_candidate
        break

if H_final is None:
    raise RuntimeError("Nie ma żadnej homografii. Uzupełnij punkty korespondencyjne albo wybierz inną klatkę.")

print(f"Wybrana homografia: {H_source}, F1={H_score:.3f}")
print(np.array2string(H_final, precision=6, suppress_small=True))

## Krok 12: obraz z nałożonym modelem boiska

To jest najważniejszy test jakościowy. Jeżeli żółte linie modelu pokrywają się z rzeczywistymi liniami boiska, `H` jest sensowne. Jeżeli overlay jest przesunięty, popraw punkty korespondencyjne.

In [ ]:
overlay = cal.draw_overlay(image, H_final, thickness=2)
show_bgr(overlay, title=f"Model boiska na obrazie, źródło H: {H_source}")

## Krok 13: warp do widoku bird view

Skoro `H` daje współrzędne w metrach, dokładamy macierz skali `S`, która zamienia metry boiska na piksele mapy z góry:

`M = S @ H`

Następnie `cv2.warpPerspective()` przenosi obraz wejściowy do prostokątnej mapy 105 m x 68 m.

In [ ]:
PIXELS_PER_METER = 10
bird, M_img_to_bird = warp_to_bird_view(image, H_final, pixels_per_meter=PIXELS_PER_METER)
bird_with_pitch = draw_pitch_on_bird(bird, pixels_per_meter=PIXELS_PER_METER, thickness=1)

show_side_by_side([
    {"img": overlay, "title": "Overlay na obrazie"},
    {"img": bird_with_pitch, "title": "Bird view po warpPerspective"},
], size=(22, 8))

## Krok 14: projekcja punktów zawodników na boisko

Dla detekcji zawodnika najczęściej rzutuje się środek dolnej krawędzi bbox, czyli przybliżony punkt styku stóp z murawą. Poniżej jest przykład z punktami klikniętymi ręcznie na obrazie.

In [ ]:
# Przykładowe punkty pikselowe. Dostosuj do stóp zawodników albo podstaw punkty z detektora.
SAMPLE_FOOT_POINTS = np.array([
    [880, 560],
    [1040, 585],
    [1505, 650],
], dtype=np.float64)

pitch_points = project_pixel_points(H_final, SAMPLE_FOOT_POINTS)

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
img_pts_vis = image.copy()
for i, (u, v) in enumerate(SAMPLE_FOOT_POINTS):
    cv2.circle(img_pts_vis, (int(u), int(v)), 8, (0, 255, 255), -1)
    cv2.putText(img_pts_vis, str(i), (int(u) + 10, int(v) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
axes[0].imshow(bgr_to_rgb(img_pts_vis))
axes[0].set_title("Punkty stóp na obrazie")
axes[0].axis("off")

ax = axes[1]
draw_pitch(ax=ax)
for i, (x_m, y_m) in enumerate(pitch_points):
    ax.plot(x_m, y_m, "o", markersize=9)
    ax.text(x_m + 1.0, y_m, str(i))
ax.set_title("Te same punkty na mapie boiska [m]")
plt.tight_layout()
plt.show()

for i, ((u, v), (x_m, y_m)) in enumerate(zip(SAMPLE_FOOT_POINTS, pitch_points)):
    print(f"Punkt {i}: image=({u:.1f}, {v:.1f}) -> pitch=({x_m:.2f}, {y_m:.2f}) m")

## Krok 15: zapis wyników

Komórka zapisuje overlay, bird view oraz macierz `H` do plików. Nazwy plików zawierają nazwę klatki i źródło homografii.

In [ ]:
OUT_DIR = Path("homography_outputs")
OUT_DIR.mkdir(exist_ok=True)

stem = image_path.stem
np.save(OUT_DIR / f"{stem}_{H_source}_H_pixel_to_pitch.npy", H_final)
cv2.imwrite(str(OUT_DIR / f"{stem}_{H_source}_overlay.jpg"), overlay)
cv2.imwrite(str(OUT_DIR / f"{stem}_{H_source}_bird_view.jpg"), bird_with_pitch)

print("Zapisano:")
print(OUT_DIR / f"{stem}_{H_source}_H_pixel_to_pitch.npy")
print(OUT_DIR / f"{stem}_{H_source}_overlay.jpg")
print(OUT_DIR / f"{stem}_{H_source}_bird_view.jpg")

## Krok 16: szybka diagnostyka wielu klatek

Ta część jest opcjonalna. Ustaw `RUN_BATCH = True`, żeby policzyć statystyki masek, odcinków i grup dla wszystkich obrazów w folderze. Pełna automatyczna estymacja może potrwać dłużej, dlatego domyślnie jej tu nie uruchamiam.

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    rows = []
    for p in IMAGE_PATHS:
        img = read_bgr(p)
        hh, ww = img.shape[:2]
        c = ClassicalHomographyV2(ww, hh, ransac_iterations=300, min_score=0.10, refine_steps=0, verbose=False)
        fm, _ = c.segment_field(img)
        lm = c.extract_line_mask(img, fm)
        sg = c.detect_segments(lm)
        mg = c.merge_collinear(sg)
        ga, gb = c.group_two_directions(mg)
        rows.append({
            "image": p.name,
            "line_pixels": int((lm > 0).sum()),
            "segments": len(sg),
            "merged_lines": len(mg),
            "group_a": len(ga),
            "group_b": len(gb),
        })
    import pandas as pd
    display(pd.DataFrame(rows))
else:
    print("Batch pominięty. Ustaw RUN_BATCH = True, jeśli chcesz policzyć diagnostykę dla wszystkich klatek.")

## Co dalej

Najbardziej praktyczny wariant dla pracy magisterskiej:

1. Traktuj pełny automat jako metodę klasyczną numer 1.
2. Traktuj wariant korespondencyjny jako metodę klasyczną numer 2 albo jako fallback dla trudnych klatek.
3. Dla sekwencji wideo zapisuj `H` tylko tam, gdzie overlay jest stabilny, a dla sąsiednich klatek rozważ propagację lub interpolację homografii.
4. Do lokalizacji zawodników rzutuj dolny środek bbox detekcji na boisko i licz błąd w metrach względem adnotacji lub ręcznie przygotowanych punktów referencyjnych.